In [ ]:
import torch
from diffusers import DiTPipeline
from tqdm import tqdm
import json
import torch.nn as nn


device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "facebook/DiT-XL-2-256"
pipe = DiTPipeline.from_pretrained(model_id, torch_dtype=torch.float16, use_safetensors=False).to(device)

# 2. Funzione di quantizzazione (stessa logica del paper)
def quantize_tensor(tensor, bits=4):
    zmin = tensor.min()
    zmax = tensor.max()
    qmax = 2**bits - 1
    alpha = (zmax-zmin)/qmax
    beta = torch.round(zmin/alpha)

    quantized = torch.round(tensor / alpha) - beta
    quantized = quantized.clamp(0, qmax)
    dequantized = (quantized + beta) * alpha
    return dequantized

# 3. Analisi della sensibilità dei layer
def analyze_sensitivity(pipe, bits=4):
    results = {}

    # Prepariamo un input fittizio (latenti e timestep) per il forward pass
    latents = torch.randn((1, 4, 64, 64), device=device, dtype=torch.float16)
    class_id = torch.tensor([0], device=device)
    timesteps = torch.arange(0, 1001, 40, device=device)
    timesteps = [t.unsqueeze(0) for t in timesteps]

    # Identifichiamo i blocchi lineari o convoluzionali
    # In una U-Net, i layer sono organizzati in down_blocks, mid_block, e up_blocks
    target_modules = []
    for i, module in enumerate(pipe.transformer.transformer_blocks):
        target_modules.append((i,module))

    block_names = []
    for name, mod in pipe.transformer.transformer_blocks[0].named_modules():
      if isinstance(mod, nn.Linear):
        block_names.append(name)
    print(block_names)
    print(f"Analisi di {len(target_modules)} moduli nel modello...")

    for id, module in tqdm(target_modules):
      for timestep in timesteps:
        timestep_value = timestep[0].item()
        with torch.no_grad():
          out_original = pipe.transformer(latents, timestep, class_labels = class_id).sample
        for block_name in block_names:
            curr_block = module.get_submodule(block_name)
        
            # Salvataggio pesi originali
            original_weight = curr_block.weight.clone()

            # 1. Output originale (FP16)
            with torch.no_grad():
                
                quantized_weight = quantize_tensor(curr_block.weight, bits=bits).detach()
                curr_block.weight.copy_(quantized_weight)
                out_quantized = pipe.transformer(latents, timestep, class_labels = class_id).sample

            # Calcolo dell'errore (MSE) sull'output finale dell'immagine latente
            error = nn.functional.mse_loss(out_original, out_quantized).item()

            if str(id) not in results:
                results[str(id)] = {}
            if str(timestep_value) not in results[str(id)]:
                results[str(id)][str(timestep_value)]={}
            
            results[str(id)][str(timestep_value)][block_name] = error

            # Ripristino pesi
            with torch.no_grad():
                curr_block.weight.copy_(original_weight)
      print(f"Layer {id} analyzed")
    return results

# 4. Esecuzione
sensitivity_results = analyze_sensitivity(pipe)

with open("errors.json","w") as F:
   json.dump(sensitivity_results,F,indent=4)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model_index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

Expected types for id2label: (dict[int, str], <class 'NoneType'>), got typing.Dict[str, str].


...

Layer 25 analyzed


 96%|█████████▋| 27/28 [08:44<00:19, 19.60s/it]

Layer 26 analyzed


100%|██████████| 28/28 [09:04<00:00, 19.44s/it]

Layer 27 analyzed
